In [ ]:
# 1️⃣ Install dependencies
!pip install flask flask-ngrok torch torchvision pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
!pip install pyngrok

In [ ]:
%%writefile app.py

Writing app.py


In [ ]:
from flask import Flask, request, jsonify
from flask_ngrok import run_with_ngrok

In [ ]:
import torch
import torch.nn as nn
import torch.quantization
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import io
from collections import OrderedDict

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyngrok import ngrok

In [ ]:
!ngrok config add-authtoken INSERTYOUROWN

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
app = Flask(__name__)

# --- Model Loading and Quantization ---
model = models.densenet121(weights=None)
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 3)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PATH = '/content/drive/MyDrive/AllModelForSkinCancer/DenseNetModelV5_3.pth'

checkpoint = torch.load(PATH, map_location=device)
state_dict = checkpoint['model_state_dict']
# strip any 'model.' prefixes
new_state_dict = OrderedDict((k[6:] if k.startswith('model.') else k, v)
                            for k, v in state_dict.items())
model.load_state_dict(new_state_dict)
model.to(device).eval()

quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
).to(device)

# --- Preprocessing & Labels ---
preprocess = transforms.Compose([
    transforms.Resize((450, 450)),
    transforms.ToTensor(),
])
lesion_map = {0: 'Normal', 1: 'Nevus', 2: 'Melanoma'}

def transform_image(image_bytes):
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    return preprocess(img).unsqueeze(0).to(device)

def get_prediction(tensor):
    with torch.no_grad():
        out = quantized_model(tensor)
        probs = torch.nn.functional.softmax(out[0], dim=0)
        idx = probs.argmax().item()
        return lesion_map.get(idx, 'Unknown'), probs[idx].item()

# --- Flask Route ---
@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        return jsonify({'error': 'No file provided'}), 400
    img_bytes = request.files['file'].read()
    tensor = transform_image(img_bytes)
    label, conf = get_prediction(tensor)
    return jsonify({'prediction': label, 'confidence': round(conf*100, 2)})

if __name__ == '__main__':
    # Expose port 5000 via ngrok
    public_url = ngrok.connect(5000)
    print(f' * ngrok tunnel running at: {public_url}')
    # Run Flask
    app.run(host='0.0.0.0', port=5000)